# EdgeDeID — Baseline Comparison & Efficiency Benchmark

Evaluates our **Qwen3-0.6B** de-identification student (SFT and SFT+GRPO) against popular
PHI/PII baselines, on the **`LLM_Eval.xlsx`** benchmark — 100 real radiology reports manually
de-identified, with the shared rule-based `preprocess_text` applied to build
the gold XML. This is the headline benchmark for the paper.

**Everything runs in-process** (`transformers` / `spaCy`) — no vLLM server. Our Qwen models
(§5) load one checkpoint at a time via `QwenDeidBaseline`, so the notebook runs top-to-bottom in
a single pass on one GPU.

| Model | Type | Source |
|---|---|---|
| Presidio | Rule-based | `presidio-analyzer` |
| GLiNER | Zero-shot NER | `urchade/gliner_mediumv2.1` |
| RoBERTa-i2b2 | Fine-tuned NER | `obi/deid_roberta_i2b2` |
| BERT-i2b2 | Fine-tuned NER | `obi/deid_bert_i2b2` |
| scispaCy | Medical NER | `en_core_web_lg` *(optional — numpy-pinned)* |
| OpenMed-PII | Fine-tuned NER | `OpenMed/OpenMed-PII-SuperClinical-Large-434M-v1` |
| OPF | Span-decode NER | `openai/privacy-filter` *(optional — from source)* |
| Qwen3-0.6B-Base | Generative (zero-shot) | `Qwen/Qwen3-0.6B-Base` |
| **EdgeDeID-SFT** | **Ours** | `EdgeDeID/Qwen3-0.6B-Base_SFT` |
| **EdgeDeID-GRPO** | **Ours** | `EdgeDeID/Qwen3-0.6B-GRPO` |

## Colab setup

Run this first **only if you're on Colab / a remote kernel**. It clones the repo, installs `edgedeid`, and prompts for your GitHub + HF tokens. On a local run it's a no-op.

In [5]:
# ==== Colab / remote setup — run me FIRST (no-op when not on Colab) ====
# Clones the repo (branch `refactor`), installs the package, and collects tokens
# via getpass so nothing secret is saved into the notebook. On a local run this
# detects "not Colab" and does nothing.
def _on_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _on_colab():
    import os, getpass
    REPO = "/content/EdgeDeID"
    if not os.path.exists(os.path.join(REPO, "pyproject.toml")):
        _gh = getpass.getpass("GitHub token (repo scope): ")
        !git clone -q --branch refactor https://{_gh}@github.com/BardiaKh/EdgeDeID.git {REPO}
        _gh = None
    %cd {REPO}
    %pip install -q -e .
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (private EdgeDeID/* checkpoints): ")
    print("Colab setup done — now run the repo-bootstrap cell below.")
else:
    print("Not on Colab — skipping remote setup (using local repo).")

/content/EdgeDeID
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.

In [4]:
# --- repo bootstrap: makes this notebook runnable from anywhere ---
import os, sys
ROOT = os.getcwd()
while ROOT != os.path.dirname(ROOT) and not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)
if not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    # installed editable (pip install -e .)? locate the repo via the package itself
    # — robust on Colab / Kaggle / remote kernels where cwd isn't inside the repo.
    try:
        import edgedeid, pathlib
        _cand = pathlib.Path(edgedeid.__file__).resolve().parent.parent
        if (_cand / "pyproject.toml").exists():
            ROOT = str(_cand)
    except Exception:
        pass
if not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = ("/content/EdgeDeID" if os.path.exists("/content/EdgeDeID/pyproject.toml")
            else "/kaggle/input/datasets/sinadaneshgar/edgedeid")  # last-resort fallbacks
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repo root:", ROOT)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/sinadaneshgar/edgedeid'

## 0 — Setup & dependencies

Repo on the path, optional installs, imports, HF auth.

> Do **not** install `scispacy` here — it pins an old numpy and breaks pandas
> (`numpy.dtype size changed`). Install it only in a throwaway session.

In [6]:
# REPO_DIR is resolved by the bootstrap cell above.
REPO_DIR = ROOT
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("REPO_DIR =", REPO_DIR)

REPO_DIR = /kaggle/input/datasets/sinadaneshgar/edgedeid


In [18]:
# Baseline dependencies. The harness skips any model whose deps are missing.
# accelerate: transformers GPU loading. Our Qwen models run in-process (no vLLM/openai needed).
%pip install -q --no-warn-conflicts accelerate gliner presidio-analyzer presidio-anonymizer spacy
!python -m spacy download en_core_web_sm

# scispaCy (optional, separate session + kernel restart):
# %pip install -q scispacy https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_web_lg-0.5.4.tar.gz

# OPF / OpenAI privacy-filter (optional, only if INCLUDE_OPF=True):
!git clone -q https://github.com/openai/privacy-filter /tmp/privacy-filter && pip install -q -e /tmp/privacy-filter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 104.5 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
fatal: destination path '/tmp/privacy-filter' already exists and is not an empty directory.


In [21]:
# ── Ensure OPF (openai/privacy-filter) is importable in THIS kernel ───────────
# Not on PyPI; installed editable from source. The editable path isn't visible to
# an already-running kernel, so add it to sys.path explicitly (avoids a restart).
import os, sys
if not os.path.isdir("/tmp/privacy-filter"):
    !git clone -q https://github.com/openai/privacy-filter /tmp/privacy-filter
%pip install -q -e /tmp/privacy-filter
if "/tmp/privacy-filter" not in sys.path:
    sys.path.insert(0, "/tmp/privacy-filter")
import opf
print("opf ready:", opf.__file__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for opf (pyproject.toml) ... done


ModuleNotFoundError: No module named 'opf'

In [22]:
!ls -la /tmp/privacy-filter
!echo "--- packaging metadata:"
!cat /tmp/privacy-filter/pyproject.toml 2>/dev/null | head -30
!cat /tmp/privacy-filter/setup.py 2>/dev/null | head -30
!echo "--- top-level python packages in the repo:"
!find /tmp/privacy-filter -maxdepth 2 -name "__init__.py"
!echo "--- is anything privacy/opf installed?:"
!pip list 2>/dev/null | grep -iE "opf|privacy"


total 76
drwxr-xr-x 6 root root  4096 Jul 26 14:35 .
drwxrwxrwt 1 root root  4096 Jul 26 14:44 ..
-rw-r--r-- 1 root root  3352 Jul 26 14:35 EVAL_AND_OUTPUT_MODES.md
drwxr-xr-x 4 root root  4096 Jul 26 14:35 examples
-rw-r--r-- 1 root root  2241 Jul 26 14:35 FINETUNING.md
drwxr-xr-x 8 root root  4096 Jul 26 14:35 .git
-rw-r--r-- 1 root root   115 Jul 26 14:35 .gitignore
-rw-r--r-- 1 root root 11357 Jul 26 14:35 LICENSE
drwxr-xr-x 8 root root  4096 Jul 26 14:35 opf
drwxr-xr-x 2 root root  4096 Jul 26 14:44 opf.egg-info
-rw-r--r-- 1 root root  1472 Jul 26 14:35 OUTPUT_SCHEMAS.md
-rw-r--r-- 1 root root   461 Jul 26 14:35 pyproject.toml
-rw-r--r-- 1 root root 12612 Jul 26 14:35 README.md
-rw-r--r-- 1 root root   492 Jul 26 14:35 SECURITY.md
--- packaging metadata:
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "opf"
version = "0.1.0"
description = "Standalone local-only OPF PyTorch model and eval tools."
readme = "README.md"
r

In [8]:
import gc, time, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch

from edgedeid.eval import compute_metrics, print_report

RESULTS_DIR = Path("artifacts/baselines"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = RESULTS_DIR / "predictions"; PRED_DIR.mkdir(parents=True, exist_ok=True)

HAS_CUDA = torch.cuda.is_available()
print("torch", torch.__version__, "| CUDA:", HAS_CUDA,
      "|", torch.cuda.get_device_name(0) if HAS_CUDA else "CPU only")

torch 2.11.0+cu128 | CUDA: True | Tesla T4


In [9]:
# ── Hugging Face auth (only needed for gated/private repos) ────────────────────
def setup_hf(token=None):
    token = token or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token is None:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if token is None:
        print("No HF token found - public repos only."); return
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")

setup_hf()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face Hub.


## 0.5 — Preflight smoke test  (run me first)

Verifies the notebook can run top-to-bottom **before** any expensive inference:
the dataset, core imports, the exact load path, and optional deps. Hard problems
raise immediately; optional gaps print a `warn` with the one-line fix.

In [11]:
import edgedeid, pathlib, os, sys
ROOT = str(pathlib.Path(edgedeid.__file__).resolve().parent.parent)
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repo root:", ROOT,
      "| xlsx present:", os.path.exists(f"{ROOT}/evals/ext_eval/LLM_Eval.xlsx"))


repo root: /content/EdgeDeID | xlsx present: True


In [12]:
# ── Preflight: fail fast, all issues at once ──────────────────────────────────
import importlib, xml.etree.ElementTree as ET

_hard, _warn = [], []
def _ok(m):  print(f"  ok   {m}")
def _bad(m): _hard.append(m); print(f"  FAIL {m}")
def _wn(m):  _warn.append(m); print(f"  warn {m}")

print("[1] dataset + load path")
DATA = os.path.join(ROOT, "evals", "ext_eval", "LLM_Eval.xlsx")
if not os.path.exists(DATA):
    _bad(f"missing dataset: {DATA}")
else:
    try:
        from edgedeid.eval.benchmark import load_radiology_benchmark
        _n, _g, _st = load_radiology_benchmark(DATA, verbose=False)
        for _v in _g:
            ET.fromstring(_v)                          # raises on malformed GT-XML
        if _st.dates_unanchored:
            _wn(f"{len(_st.dates_unanchored)} gold date(s) not found in their note")
        _ok(f"{len(_n)} rows · loader valid · {_st.dates_reanchored} gold dates re-anchored")
    except Exception as e:
        _bad(f"load path broken: {type(e).__name__}: {e}")
print("[2] core imports")
for _mod, _sym in [("edgedeid.eval","compute_metrics"), ("edgedeid.eval","print_report")]:
    try: getattr(importlib.import_module(_mod), _sym); _ok(f"{_mod}.{_sym}")
    except Exception as e: _bad(f"{_mod}.{_sym}: {e}")

print("[3] baseline wrappers + our model (import only, no weights)")
try:
    from edgedeid.baselines import (PresidioBaseline, GLiNERBaseline, RobertaDeidBaseline,
        ScispaCyBaseline, BertDeidBaseline, OpenMedPIIBaseline, OPFBaseline,
        StanfordDeidBaseline, QwenDeidBaseline)
    from edgedeid.eval.metrics import _extract_phi_block
    _ok("8 baseline classes + QwenDeidBaseline + _extract_phi_block")
except Exception as e:
    _bad(f"baseline import: {e}")

print("[4] optional deps")
try: _ok(f"torch CUDA={HAS_CUDA}")
except NameError: _wn("HAS_CUDA undefined — run the §0 setup cell first")
try: import openai; _ok("openai (vLLM client for §5)")
except Exception: _wn("openai missing → §5 our-model eval unavailable (pip install openai)")
def _ensure_spacy(model="en_core_web_sm"):
    """Install spaCy + the model on first run so QC Tier-3 is available."""
    import importlib, subprocess, sys
    try:
        import spacy
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy"], check=True)
        importlib.invalidate_caches(); import spacy
    try:
        spacy.load(model)
    except OSError:
        print(f"  ... downloading spaCy {model} (one-time)")
        subprocess.run([sys.executable, "-m", "spacy", "download", model], check=True)
        importlib.invalidate_caches(); spacy.load(model)
    return True
try:
    _ensure_spacy(); _ok("spaCy en_core_web_sm ready (QC Tier-3)")
except Exception as e:
    _wn(f"spaCy en_core_web_sm unavailable ({type(e).__name__}) → QC Tier-3 degrades to Tier-2")
if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
    _ok("HF token present (private EdgeDeID/* checkpoints)")
else:
    _wn("no HF_TOKEN → private EdgeDeID/* checkpoints will 401 (run §0 setup_hf)")

print("\n" + "=" * 60)
if _hard:
    print(f"PREFLIGHT FAILED — {len(_hard)} hard issue(s); fix before running top-to-bottom:")
    for _m in _hard: print("   -", _m)
    raise RuntimeError("preflight hard failures: " + "; ".join(_hard))
print(f"PREFLIGHT PASSED  ({len(_warn)} warning(s) — see 'warn' lines above)")

[1] dataset + load path
Starting text processing pipeline...
Fixing spaces in closing tags...
Fixing spaces in tags...
Processing phone numbers...
Processing dates...
Processing ages...
Correcting gender terms...
Processing name tags...
Simplifying name tags...
Reformatting ID tags...
Processing remaining dates...
Removing markdown tags...

Processing Summary:
  Entity       Detected  Reformatted
  ----------  ---------  -----------
  name               18           15
  age                 3            -
  date                9            3
  place               3            -
  address             2            -
  phone               1            -
  email               1            -
  id                  1            -
  url                 4            -
  device              1            -

Text processing pipeline completed!
  ok   100 rows · preprocess_text + convert_to_xml valid
[2] core imports
  ok   edgedeid.eval.compute_metrics
  ok   edgedeid.eval.print_report
[3] baselin

## 1 — Load the external test set (`LLM_Eval.xlsx`)

The 100 manually-annotated radiology reports. `preprocess_text` applies the rule-based
normalisation and `convert_to_xml` builds the gold `<phi_entities>` XML — exactly the pipeline
the supervisor's `en_eval.ipynb` uses, so the gold labels are identical.

The gold `<date>` may show `<formatted>2024-??-??</formatted>`: intentional, not a bug — when the
source states only a partial date (e.g. year), the unknown parts are filled with `?`.

In [13]:
# Radiology benchmark: the clean 100-note set (evals/ext_eval/LLM_Eval.xlsx), loaded
# with misspelled gold tags repaired and gold dates re-anchored to the note's own
# wording -- see edgedeid/eval/benchmark.py for why revizied_evals.xlsx is not used.
from edgedeid.eval.benchmark import load_radiology_benchmark

notes, gt_xmls, BENCH_STATS = load_radiology_benchmark(f"{ROOT}/evals/ext_eval/LLM_Eval.xlsx")


Starting text processing pipeline...
Fixing spaces in closing tags...
Fixing spaces in tags...
Processing phone numbers...
Processing dates...
Processing ages...
Correcting gender terms...
Processing name tags...
Simplifying name tags...
Reformatting ID tags...
Processing remaining dates...
Removing markdown tags...

Processing Summary:
  Entity       Detected  Reformatted
  ----------  ---------  -----------
  name              542          449
  age                73            -
  date              271           56
  place              74            -
  address            67            -
  phone              71            -
  email              62            -
  id                 63            -
  url               115            -
  device              6            -

Text processing pipeline completed!
100 evaluation samples (LLM_Eval.xlsx)


In [14]:
# Sanity-check one sample.
print("NOTE:\n", notes[0][:400], "\n")
print("GOLD XML:\n", gt_xmls[0][:400])

NOTE:
 Amit Kumar, a 35-year-old male, visited the emergency department at NYU Langone Medical Center, complaining of severe chest pain and shortness of breath. Residence: 3456 Broadway, Apt 2A, New York, NY 10024.

Dr. Rachel Lee performed an ECG and ordered a CT scan with contrast to rule out pulmonary embolism. Results were negative for PE but showed signs of pneumonia in the lower left lobe. Mr. Kuma 

GOLD XML:
 <phi_entities>
<names>
<name id="1" type="patient">
<variations>
<value>Amit Kumar</value>
<value>Kumar</value>
</variations>
</name>
<name id="2" type="provider">
<variations>
<value>Rachel Lee</value>
</variations>
</name>
</names>
<ages>
<age>35</age>
</ages>
<dates/>
<places>
<place id="1" type="hospital">NYU Langone Medical Center</place>
</places>
<addresses>
<address>3456 Broadway, Apt 2A, 


## 2 — Baseline registry + helpers

A lazy `(label, factory)` registry of the **in-process** baselines (our generative Qwen models
run in §5). Nothing loads until a loop calls the factory, so one model occupies the GPU at a
time. Plus small single-purpose helpers.

In [15]:
from edgedeid.baselines import (
    PresidioBaseline, GLiNERBaseline, RobertaDeidBaseline, ScispaCyBaseline,
    BertDeidBaseline, OpenMedPIIBaseline, OPFBaseline, StanfordDeidBaseline,
)

INCLUDE_SCISPACY = True    # needs en_core_web_lg (numpy-pinned; install separately)
INCLUDE_OPF      = True    # needs openai/privacy-filter from source

BASELINES = [
    ("Presidio",     lambda: PresidioBaseline(score_threshold=0.5)),
    ("GLiNER",       lambda: GLiNERBaseline(model_name="urchade/gliner_mediumv2.1", threshold=0.5)),
    ("RoBERTa-i2b2", lambda: RobertaDeidBaseline(model_name="obi/deid_roberta_i2b2")),
    ("BERT-i2b2",    lambda: BertDeidBaseline(model_name="obi/deid_bert_i2b2")),
    ("OpenMed-PII",  lambda: OpenMedPIIBaseline(model_name="OpenMed/OpenMed-PII-SuperClinical-Large-434M-v1")),
    ("Stanford-deid", lambda: StanfordDeidBaseline(model_name="StanfordAIMI/stanford-deidentifier-base")),
]
if INCLUDE_SCISPACY:
    BASELINES.append(("scispaCy", lambda: ScispaCyBaseline(model_name="en_core_web_lg")))
if INCLUDE_OPF:
    BASELINES.append(("OPF", lambda: OPFBaseline()))

print(f"{len(BASELINES)} baselines registered:", [b for b, _ in BASELINES])

8 baselines registered: ['Presidio', 'GLiNER', 'RoBERTa-i2b2', 'BERT-i2b2', 'OpenMed-PII', 'Stanford-deid', 'scispaCy', 'OPF']


In [16]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def count_params(model):
    # Best-effort parameter count for any wrapper (None if unknown).
    def _n(o):
        if hasattr(o, "num_parameters"):
            try: return int(o.num_parameters())
            except Exception: pass
        if hasattr(o, "parameters"):
            try: return sum(p.numel() for p in o.parameters())
            except Exception: pass
        return None
    for attr in ("model", "_model"):
        if hasattr(model, attr) and (n := _n(getattr(model, attr))): return n
    if hasattr(model, "_pipe") and (n := _n(getattr(model._pipe, "model", None))): return n
    return None

def on_gpu(model):
    return str(getattr(model, "device", "cpu")).startswith("cuda")

def free_model(model):
    # Drop references and release GPU memory before the next model loads.
    for attr in ("model", "_model", "_pipe", "_opf", "_analyzer", "_nlp"):
        if hasattr(model, attr):
            try: setattr(model, attr, None)
            except Exception: pass
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def evaluate(preds, supported=None):
    # compute_metrics scoped to what the model can emit (None = full schema).
    return compute_metrics(preds, gt_xmls, source_notes=notes, supported_entities=supported)

def result_row(name, r, n_params, sec_per_note):
    return {
        "Model": name,
        "Micro-P": round(r.micro_precision, 4), "Micro-R": round(r.micro_recall, 4),
        "Micro-F1": round(r.micro_f1, 4), "F-b(2)": round(r.f_beta, 4),
        "Macro-F1": round(r.macro_f1, 4), "Zero-leak": round(r.zero_leak_rate, 4),
        "XML Parse": round(r.xml_parse_rate, 4),
        "Halluc.": round(r.hallucination_rate, 4),
        "Token Reduction": (round(r.token_reduction_rate, 4)
                            if r.token_reduction_rate == r.token_reduction_rate else None),
        "#Params (M)": round(n_params / 1e6, 1) if n_params else None,
        "Sec/Note": round(sec_per_note, 4) if sec_per_note is not None else None,
    }

def per_entity_frame(results):
    tags = sorted({t for r in results.values() for t in r.per_entity})
    return pd.DataFrame([
        {"Entity": e, **{name: (round(r.per_entity[e].f1, 4) if e in r.per_entity else None)
                         for name, r in results.items()}}
        for e in tags
    ]).set_index("Entity")

print("helpers ready")

helpers ready


## 3 — Smoke test

Builds each baseline and checks it returns parseable `<phi_entities>` XML on 2 notes — catches
missing deps / bad checkpoints in seconds instead of after a full pass.

In [24]:
from edgedeid.eval.metrics import _extract_phi_block

SMOKE_NOTES = notes[:2]
for label, factory in BASELINES:
    try:
        m = factory()
    except Exception as e:
        print(f"  x {label}: load failed - {type(e).__name__}: {e}"); continue
    try:
        out = m.predict_xml(SMOKE_NOTES[0])
        ok = isinstance(out, str) and _extract_phi_block(out) is not None
        print(f"  {'ok' if ok else 'x '} {label}: {'parseable XML' if ok else 'NO valid <phi_entities>'}")
    finally:
        free_model(m)

  ok Presidio: parseable XML


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


  ok GLiNER: parseable XML


Device set to use cuda:0


  ok RoBERTa-i2b2: parseable XML


... (11 outputs left out)

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

viterbi_calibration.json:   0%|          | 0.00/372 [00:00<?, ?B/s]


Download complete: /root/.opf/privacy_filter
------


  ok OPF: parseable XML


### 3.5 — Smoke inference (5 notes)

Runs the **full predict → score** path (identical to §4) on just 5 notes for every registered baseline. Fast sanity check that inference *and* metric computation work end-to-end before committing to the full 100-note run. Prints per-model F1/recall/parse; load or scoring failures are caught and reported, never fatal.

In [25]:
# ── Smoke inference: full predict→score path on 5 notes (fast preflight for §4) ──
from edgedeid.eval import compute_metrics

SMOKE_N = 5
smoke_notes, smoke_gts = notes[:SMOKE_N], gt_xmls[:SMOKE_N]

print(f"Smoke inference on {SMOKE_N} notes for {len(BASELINES)} baselines\n")
for label, factory in BASELINES:
    try:
        m = factory()
    except Exception as e:
        print(f"  x  {label:14s} load failed: {type(e).__name__}: {e}"); continue
    try:
        preds = [m.predict_xml(n) for n in smoke_notes]
        r = compute_metrics(preds, smoke_gts, source_notes=smoke_notes,
                            supported_entities=m.supported_entities)
        print(f"  ok {label:14s} F1={r.micro_f1:.3f}  R={r.micro_recall:.3f}  "
              f"parse={r.xml_parse_rate:.2f}")
    except Exception as e:
        print(f"  x  {label:14s} inference/score failed: {type(e).__name__}: {e}")
    finally:
        free_model(m)

print(f"\nSmoke inference done — if these look sane, run §4 for the full set.")

Smoke inference on 5 notes for 8 baselines



  ok Presidio       F1=0.736  R=0.688  parse=1.00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


  ok GLiNER         F1=0.786  R=0.688  parse=1.00


... (4 outputs left out)

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


  ok OpenMed-PII    F1=0.642  R=0.516  parse=1.00


Device set to use cuda:0


  ok Stanford-deid  F1=0.404  R=0.359  parse=1.00
  ok scispaCy       F1=0.668  R=0.594  parse=1.00
  ok OPF            F1=0.815  R=0.688  parse=1.00

Smoke inference done — if these look sane, run §4 for the full set.


## 4 — Run baselines  (accuracy + latency)

Explicit loop: load -> sequential `predict_xml` -> metrics -> free. Sequential timing is the
realistic per-note latency for these models. Writes per-model predictions to `results/predictions/`.

In [ ]:
baseline_results, baseline_rows = {}, {}
for label, factory in BASELINES:
    print(f"\n{'='*70}\n> {label}\n{'='*70}")
    try:
        model = factory()
    except Exception as e:
        import traceback; print(f"Skipping {label}: {type(e).__name__}: {e}"); traceback.print_exc(); continue

    preds, t0 = [], time.perf_counter()
    for note in tqdm(notes, desc=label):
        preds.append(model.predict_xml(note))
    spn = (time.perf_counter() - t0) / max(len(notes), 1)

    r = evaluate(preds, supported=model.supported_entities)
    baseline_results[label] = r
    baseline_rows[label] = result_row(label, r, count_params(model), spn)
    print_report(r, title=label)
    print(f"   {spn*1000:.1f} ms/note  ·  {baseline_rows[label]['#Params (M)']} M params")
    pd.Series(preds, name="prediction").to_csv(PRED_DIR / f"pred_{label}.csv", index=False)
    free_model(model)

print("\nBaselines done:", list(baseline_results))

## 5 — Our Qwen models  (accuracy + QC ablation)

Loads each checkpoint in `OUR_CHECKPOINTS` **in-process** via `QwenDeidBaseline`
(transformers) and runs greedy batched inference — no vLLM server. **Resumable**:
a checkpoint whose `predictions/pred_<name>.csv` already exists is reused and not
re-run.

For the checkpoints in `QC_ABLATION_FROM` it then derives a **`<name>+QC`** row by
running the layered QC pipeline (`apply_qc_batch`) over the *same* predictions — no
extra inference. This produces the SFT vs SFT+QC vs GRPO vs GRPO+QC ablation in one
pass, alongside the baselines. Tier-3 QC is enabled automatically iff the spaCy
model is installed (`python -m spacy download en_core_web_sm`).

In [ ]:
# ── §5 config: our checkpoints + QC ablation (in-process, no vLLM) ────────────
from edgedeid.baselines import QwenDeidBaseline
from edgedeid.qc import apply_qc_batch, QCConfig

OUR_BATCH = 16   # notes per forward pass for greedy batched inference

# (HF id or local path, display name, params in millions). Point the id at a local
# directory if you have the weights, to avoid re-downloading.
OUR_CHECKPOINTS = [
    ("Qwen/Qwen3-0.6B-Base",         "Qwen3-0.6B-Base", 596.0),   # zero-shot
    ("EdgeDeID/Qwen3-0.6B-Base_SFT", "EdgeDeID-SFT",    596.0),
    ("EdgeDeID/Qwen3-0.6B-GRPO",     "EdgeDeID-GRPO",   596.0),
]

# Checkpoints that also get a +QC ablation row (post-processing, no extra inference).
QC_ABLATION_FROM = {"EdgeDeID-SFT", "EdgeDeID-GRPO"}

# Tier-3 (name/place/address NER) only if the spaCy model is present; otherwise
# QC runs Tier-1 + Tier-2 and the ablation still works, just without Tier-3.
try:
    import spacy
    spacy.load("en_core_web_sm")
    _TIER3 = True
except Exception:
    _TIER3 = False
    print("Tier-3 disabled — run `python -m spacy download en_core_web_sm` to enable "
          "name/place/address QC.")
QC_CONFIG = QCConfig(use_tier3=_TIER3)
print("Tier-3 QC:", _TIER3, "| checkpoints:", [n for _, n, _ in OUR_CHECKPOINTS])

In [ ]:
# ── run our checkpoints in-process (resumable) + derive +QC ablation rows ─────
our_results = globals().get("our_results", {})
our_rows    = globals().get("our_rows", {})

def _preds_path(name):
    return PRED_DIR / f"pred_{name}.csv"

def _load_or_infer(model_id, name):
    """Predictions for `name`, reusing the saved CSV if present, else loading the
    checkpoint in-process (QwenDeidBaseline) and running greedy batched inference."""
    p = _preds_path(name)
    if p.exists():
        print(f"  reuse cached predictions: {p.name}")
        return pd.read_csv(p)["prediction"].fillna("").tolist(), None
    model = QwenDeidBaseline(model_id, name=name)
    try:
        t0 = time.perf_counter()
        preds = []
        for i in tqdm(range(0, len(notes), OUR_BATCH), desc=name):
            preds.extend(model.predict_xml_batch(notes[i:i + OUR_BATCH]))
        spn = (time.perf_counter() - t0) / max(len(notes), 1)
    finally:
        free_model(model)
    pd.Series(preds, name="prediction").to_csv(p, index=False)
    return preds, spn

def _record(name, preds, params_m, spn):
    r = evaluate(preds, supported=None)          # our models can emit the full schema
    our_results[name] = r
    our_rows[name] = result_row(name, r, int(params_m * 1e6), spn)
    print_report(r, title=name)

for model_id, name, params_m in OUR_CHECKPOINTS:
    print(f"\n{'='*70}\n> {name}  ({model_id})\n{'='*70}")
    try:
        preds, spn = _load_or_infer(model_id, name)
    except Exception as e:
        import traceback; print(f"  x skipped {name}: {type(e).__name__}: {e}"); traceback.print_exc()
        continue
    _record(name, preds, params_m, spn)

    if name in QC_ABLATION_FROM:                 # +QC row from the same predictions
        qc_name = f"{name}+QC"
        finals, _ = apply_qc_batch(preds, notes, config=QC_CONFIG)
        pd.Series(finals, name="prediction").to_csv(_preds_path(qc_name), index=False)
        _record(qc_name, finals, params_m, None) # QC latency is negligible vs inference

print("\nOur models + QC ablation done:", list(our_results))

## 6 — Combined comparison table

Merges baselines (§4) and our models (§5) into one table + per-entity F1, saved to CSV.

In [ ]:
all_results = {**globals().get("baseline_results", {}), **globals().get("our_results", {})}
all_rows    = {**globals().get("baseline_rows", {}),    **globals().get("our_rows", {})}
assert all_rows, "No results collected — run §4 (baselines) and/or §5 (our models) first."

comparison_df = pd.DataFrame(list(all_rows.values())).set_index("Model")
comparison_df.to_csv(RESULTS_DIR / "comparison.csv")

entity_df = per_entity_frame(all_results)
entity_df.to_csv(RESULTS_DIR / "per_entity.csv")
with open(RESULTS_DIR / "results.pkl", "wb") as f:
    pickle.dump(all_results, f)

print(f"Saved comparison.csv ({len(all_rows)} models), per_entity.csv, results.pkl "
      f"-> {RESULTS_DIR}")
comparison_df

## 7 — Accuracy plots

Saved as PNGs in `results/`.
1. Per-entity **F1** heatmap (grey = entity the model can't emit).
2. Per-entity **recall** heatmap (the safety / leak view).
3. **Precision–Recall** scatter with F1 iso-curves.
4. **Headline** metric bars.
5. **Params vs F1** efficiency frontier (* = ours).

In [ ]:
RESULTS = all_results
model_names = list(RESULTS)
ALL_TAGS = ["name", "age", "date", "place", "address", "phone", "email", "id", "url", "device"]

def _entity_matrix(attr):
    M = np.full((len(model_names), len(ALL_TAGS)), np.nan)
    for i, mn in enumerate(model_names):
        pe = RESULTS[mn].per_entity
        for j, t in enumerate(ALL_TAGS):
            if (m := pe.get(t)) is not None:
                M[i, j] = getattr(m, attr)
    return M

def _heatmap(M, title, fname, cmap_name):
    cmap = plt.get_cmap(cmap_name).copy(); cmap.set_bad("0.85")
    fig, ax = plt.subplots(figsize=(11, 1.6 + 0.5 * len(model_names)))
    im = ax.imshow(M, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(ALL_TAGS))); ax.set_xticklabels(ALL_TAGS, rotation=40, ha="right")
    ax.set_yticks(range(len(model_names))); ax.set_yticklabels(model_names)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            ax.text(j, i, "N/A" if np.isnan(v) else f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color="0.4" if np.isnan(v) else ("white" if v < 0.5 else "black"))
    ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    plt.tight_layout(); plt.savefig(RESULTS_DIR / fname, dpi=150, bbox_inches="tight"); plt.show()

_heatmap(_entity_matrix("f1"), "Per-entity F1  (grey = unsupported by model)",
         "plot_f1_heatmap.png", "viridis")

In [ ]:
_heatmap(_entity_matrix("recall"), "Per-entity recall  (red = PHI leaking, green = caught)",
         "plot_recall_heatmap.png", "RdYlGn")

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 6.8))
pp = np.linspace(0.001, 1, 400)
for f1 in (0.2, 0.4, 0.6, 0.8):
    rr = np.where(2*pp - f1 > 0, f1*pp/(2*pp - f1), np.nan); rr[(rr <= 0) | (rr > 1)] = np.nan
    ax.plot(pp, rr, "--", color="0.7", lw=0.7)
    ax.text(0.985, f1*0.985/(2*0.985 - f1), f"F1={f1}", color="0.5", fontsize=7, va="bottom", ha="right")
for mn in model_names:
    r = RESULTS[mn]
    ax.scatter(r.micro_precision, r.micro_recall, s=90, zorder=3)
    ax.annotate(mn, (r.micro_precision, r.micro_recall), textcoords="offset points", xytext=(6, 5), fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.set_xlabel("Micro precision"); ax.set_ylabel("Micro recall")
ax.set_title("Precision-Recall trade-off (micro, F1 iso-curves)"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "plot_pr_scatter.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
headline = [("Micro-F1", "micro_f1"), ("Micro-Recall", "micro_recall"),
            ("Zero-leak", "zero_leak_rate"), ("Macro-F1", "macro_f1")]
x = np.arange(len(headline)); w = 0.8 / max(len(model_names), 1)
fig, ax = plt.subplots(figsize=(10, 4.8))
for i, mn in enumerate(model_names):
    bars = ax.bar(x + i*w, [getattr(RESULTS[mn], a) for _, a in headline], w, label=mn)
    ax.bar_label(bars, fmt="%.2f", fontsize=6, padding=2)
ax.set_xticks(x + w*(len(model_names)-1)/2); ax.set_xticklabels([n for n, _ in headline])
ax.set_ylim(0, 1.05); ax.set_ylabel("score"); ax.set_title("Headline metrics by model")
ax.legend(fontsize=7, ncol=2); plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_headline_bars.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for mn in model_names:
    p = comparison_df.loc[mn, "#Params (M)"] if mn in comparison_df.index else None
    f1 = RESULTS[mn].micro_f1
    if p and p == p:
        ours = mn.startswith("EdgeDeID")
        ax.scatter(p, f1, s=140 if ours else 80, marker="*" if ours else "o",
                   zorder=3, edgecolor="k" if ours else "none")
        ax.annotate(mn, (p, f1), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("Parameters (M, log)"); ax.set_ylabel("Micro-F1")
ax.set_title("Accuracy vs model size  (* = ours)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "plot_params_vs_f1.png", dpi=150, bbox_inches="tight"); plt.show()

## 8 — Latency benchmark  (apples-to-apples, edge-realistic)

Every model is timed through the **same** protocol: single note at a time (no
server batching), on the **same device**, after warm-up, reporting the median.
This is the deployment-realistic number — one report arrives, one report is
de-identified — and it is directly comparable across models because they all go
through `predict_xml`.

`LAT_DEVICE = "cpu"` is the number to report for an edge claim (set `"cuda"` for
GPU). Note this is **not** the throughput seen in §4/§5, which is batched /
server-side and not comparable; `latency.csv` written here is the authoritative
latency table.

The quantized **GGUF via llama.cpp** path (optional cell below) is the true
deployment runtime for our student and is the headline edge-cost figure.

In [ ]:
# ── §8 config + timing helpers ───────────────────────────────────────────────
import statistics
try:
    import psutil
    _PROC = psutil.Process()
except Exception:
    _PROC = None

LAT_DEVICE = "cpu"          # "cpu" = edge-realistic headline; "cuda" for GPU numbers
LAT_N      = 30             # notes sampled for timing
LAT_WARMUP = 3              # untimed calls first (JIT, allocator, tokenizer cache)
LAT_SUBSET = notes[:LAT_N]

# Force every model onto CPU for a fair edge comparison (must precede model load).
if LAT_DEVICE == "cpu":
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

def _rss_mb():
    return _PROC.memory_info().rss / 1e6 if _PROC else None

def time_predict(predict_fn, subset=LAT_SUBSET, warmup=LAT_WARMUP):
    """Median / p90 single-note latency (ms) and throughput for one model."""
    for n in subset[:warmup]:
        predict_fn(n)
    per_ms = []
    for n in subset:
        t0 = time.perf_counter()
        predict_fn(n)
        per_ms.append((time.perf_counter() - t0) * 1000)
    per_ms.sort()
    med = statistics.median(per_ms)
    return {
        "ms/note (median)": round(med, 1),
        "ms/note (p90)":    round(per_ms[max(int(0.9 * len(per_ms)) - 1, 0)], 1),
        "notes/sec":        round(1000 / med, 2) if med else None,
    }

print(f"Latency protocol: device={LAT_DEVICE}, n={LAT_N}, warmup={LAT_WARMUP}, "
      f"psutil={'yes' if _PROC else 'no (RSS omitted)'}")

In [ ]:
# ── time every model uniformly through predict_xml ────────────────────────────
from edgedeid.baselines import QwenDeidBaseline

# baselines + our checkpoints via the transformers wrapper (NOT vLLM), so all
# models share one protocol. GGUF deployment runtime is timed separately below.
LAT_MODELS = list(BASELINES) + [
    (name, (lambda mid=mid, nm=name: QwenDeidBaseline(mid, name=nm)))
    for mid, name, _ in OUR_CHECKPOINTS
]

latency_rows = {}
for label, factory in LAT_MODELS:
    print(f"> {label}")
    try:
        rss0 = _rss_mb()
        m = factory()
        if LAT_DEVICE == "cpu" and hasattr(m, "model") and hasattr(getattr(m, "model"), "to"):
            m.model.to("cpu", dtype=torch.float32)
            m.device = "cpu"
        rss1 = _rss_mb()
    except Exception as e:
        print(f"  x skipped: {type(e).__name__}: {e}")
        continue
    try:
        row = time_predict(m.predict_xml)
        row["Peak RSS (MB)"] = round(rss1 - rss0) if (rss0 and rss1) else None
        row["#Params (M)"]   = (round((count_params(m) or 0) / 1e6, 1) or None)
        row["Device"]        = LAT_DEVICE
        row["Runtime"]       = "transformers"
        latency_rows[label] = row
        print(f"   {row['ms/note (median)']} ms/note  ·  {row['notes/sec']} notes/s"
              f"  ·  {row['Peak RSS (MB)']} MB")
    finally:
        free_model(m)

lat_df = pd.DataFrame(latency_rows).T
lat_df.index.name = "Model"
lat_df.to_csv(RESULTS_DIR / "latency.csv")
print("\nsaved latency.csv")
lat_df

### 8.1 — Deployment runtime: quantized GGUF via llama.cpp  (optional)

The **true edge-deployment cost** for our student: the Q4_K_M GGUF run on CPU
through `llama.cpp`, not the `transformers` path above. Set `GGUF_MODEL` and
`LLAMA_CLI` to your local build. This is the figure to headline in the paper.

In [ ]:
# ── deployment-runtime latency: Q4_K_M GGUF on CPU via llama.cpp ──────────────
import subprocess
from edgedeid.eval.inference import _build_prompt

GGUF_MODEL = None   # e.g. "artifacts/gguf/Qwen3-0.6B-Base_SFT-Q4_K_M.gguf"
LLAMA_CLI  = "llama.cpp/build/bin/llama-cli"

if GGUF_MODEL:
    def gguf_predict(note):
        subprocess.run(
            [LLAMA_CLI, "-m", GGUF_MODEL, "-p", _build_prompt(note),
             "-n", "1024", "--temp", "0", "-no-cnv", "-st"],
            capture_output=True, text=True, timeout=300,
        )
    row = time_predict(gguf_predict)
    row.update({"Peak RSS (MB)": None, "#Params (M)": 596.0,
                "Device": "cpu", "Runtime": "llama.cpp Q4_K_M"})
    latency_rows["EdgeDeID (GGUF)"] = row
    pd.DataFrame(latency_rows).T.rename_axis("Model").to_csv(RESULTS_DIR / "latency.csv")
    print(f"GGUF: {row['ms/note (median)']} ms/note · {row['notes/sec']} notes/s")
else:
    print("Set GGUF_MODEL to time the quantized deployment runtime "
          "(the headline edge-latency figure).")

## 9 — Master table

Sorts by micro-F1 and exports **CSV** + **LaTeX** (`results/master_table.tex`).

In [ ]:
master = pd.read_csv(RESULTS_DIR / "comparison.csv").set_index("Model")

# Merge authoritative single-note latency (from §8) over the incidental Sec/Note.
if (RESULTS_DIR / "latency.csv").exists():
    lat = pd.read_csv(RESULTS_DIR / "latency.csv").set_index("Model")
    for col in ["ms/note (median)", "notes/sec", "Peak RSS (MB)"]:
        if col in lat.columns:
            master[col] = lat[col]

master = master.sort_values("Micro-F1", ascending=False)
master.to_csv(RESULTS_DIR / "master_table.csv")

paper_cols = [c for c in ["Micro-P", "Micro-R", "Micro-F1", "Macro-F1", "Zero-leak",
                          "Halluc.", "#Params (M)", "ms/note (median)", "notes/sec"]
              if c in master.columns]
master[paper_cols].to_latex(
    RESULTS_DIR / "master_table.tex", float_format="%.3f", bold_rows=True, na_rep="--",
    caption="PHI de-identification on LLM_Eval.xlsx (clean 100-note benchmark): accuracy, size and edge latency.",
    label="tab:results")
print("Saved master_table.csv and master_table.tex")

best = master.index[0]
print(f"\nBest micro-F1: {best} ({master.loc[best, 'Micro-F1']:.3f})")
master[paper_cols]